# Lab 04 — Signature Detection in Documents (Conditional DETR)

> **GraphoLab** | Forensic Graphology Laboratory

**Model:** `tech4humans/conditional-detr-50-signature-detector` (Hugging Face / Apache 2.0)  
**Task:** Locate and extract signatures from scanned documents  
**Forensic use case:** Document pipeline — detect → extract → verify (feeds into Lab 03)

---

## How Conditional DETR Object Detection Works

DETR (DEtection TRansformer) reformulates object detection as a set-prediction problem using a Transformer encoder-decoder:

1. A **CNN backbone** (ResNet-50) extracts spatial feature maps from the image.
2. The **Transformer encoder** refines the feature maps with self-attention.
3. A fixed set of learned **object queries** are decoded by the Transformer decoder — each query "attends" to the relevant image region.
4. A feed-forward network predicts a bounding box and class label for each query.

**Conditional DETR** (Meng et al., 2021) speeds up convergence vs. original DETR by conditioning cross-attention on predicted reference points, reducing the number of training epochs needed by ~10×.

The model was fine-tuned on annotated signature images to detect signatures specifically in document scans, achieving **mAP@50 = 93.65%**.


## GraphoLab Core — Quick Start

> The production implementation of signature detection is available in [`core/signature.py`](../core/signature.py).
> It wraps **Conditional DETR** (`tech4humans/conditional-detr-50-signature-detector`) with lazy thread-safe model loading,
> bounding-box annotation, and automatic cropping of detected signatures.
>
> Run the cell below to import it directly. The remaining cells implement the same detection
> pipeline from scratch for educational purposes.

In [ ]:
# GraphoLab Core — production usage
# Run this cell to use the shared core module instead of the notebook implementation below.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from core.signature import detect_and_crop, sig_detect, get_detector
from PIL import Image
import numpy as np

# Example: detect and crop signature from a document
# doc = np.array(Image.open("../data/samples/document_with_signature_01.png").convert("RGB"))
# annotated, crop, summary = detect_and_crop(doc)
# print(summary)
print("core.signature imported — detect_and_crop(), sig_detect() ready.")

## Setup


In [ ]:
# !pip install transformers torch Pillow matplotlib opencv-python

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from transformers import AutoImageProcessor, AutoModelForObjectDetection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Libraries loaded. Device: {DEVICE}")

## Load the Model

The model is downloaded from Hugging Face the first time and cached locally (~170 MB).  
No token or access request required — Apache 2.0 licence, publicly available.


In [ ]:
DETR_REPO = "tech4humans/conditional-detr-50-signature-detector"

print("Loading Conditional DETR signature detector from Hugging Face...")
processor = AutoImageProcessor.from_pretrained(DETR_REPO)
model = AutoModelForObjectDetection.from_pretrained(DETR_REPO).to(DEVICE).eval()
print("Model ready.")

## Helper Functions


In [ ]:
def detect_signatures(image: Image.Image | Path | str, conf_threshold: float = 0.3) -> dict:
    """Run signature detection on a PIL image or image path."""
    if not isinstance(image, Image.Image):
        image = Image.open(image).convert("RGB")
    else:
        image = image.convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(
        outputs, threshold=conf_threshold, target_sizes=target_sizes
    )[0]

    detections = []
    for score, box in zip(results["scores"], results["boxes"]):
        x1, y1, x2, y2 = box.cpu().numpy().astype(int)
        detections.append({
            "bbox": (x1, y1, x2, y2),
            "confidence": float(score.cpu()),
        })

    return {
        "image": image,
        "detections": detections,
        "count": len(detections),
    }


def show_detections(result: dict, title: str = "Signature Detection") -> None:
    """Visualise detected signatures with bounding boxes."""
    image = result["image"]
    detections = result["detections"]

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det["bbox"]
        conf = det["confidence"]
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 6, f"Signature #{i+1}  {conf:.0%}",
                color='red', fontsize=9, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.6, pad=1))

    ax.set_title(f"{title} — {len(detections)} signature(s) found", fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def crop_signatures(result: dict, output_dir: Path) -> list:
    """Crop and save each detected signature as a separate image file."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    image = result["image"]
    saved = []
    for i, det in enumerate(result["detections"]):
        x1, y1, x2, y2 = det["bbox"]
        crop = image.crop((x1, y1, x2, y2))
        out_path = output_dir / f"detected_signature_{i+1:02d}.png"
        crop.save(out_path)
        saved.append(out_path)
        print(f"  Saved: {out_path}")
    return saved

## Demo 1 — Detect Signatures in a Sample Document


In [ ]:
def make_synthetic_document() -> Path:
    """Create a simple synthetic document image with a fake signature for demo."""
    from PIL import ImageDraw
    img = Image.new('RGB', (800, 1000), color='white')
    draw = ImageDraw.Draw(img)

    # Simulate document text lines
    for y in range(60, 700, 35):
        line_width = np.random.randint(300, 700)
        draw.rectangle([(60, y), (60 + line_width, y + 10)], fill='#cccccc')

    # Signature area label
    draw.text((60, 750), "Signature:", fill='black')
    # Simulate a cursive signature stroke
    points = [(200, 800), (220, 780), (260, 820), (300, 790),
              (340, 810), (380, 780), (400, 805)]
    draw.line(points, fill='black', width=3)
    draw.line([(200, 830), (420, 830)], fill='black', width=1)

    out = Path("../data/samples/document_with_signature_demo.png")
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out)
    return out


samples_dir = Path("../data/samples")
real_doc = samples_dir / "document_with_signature_01.png"

if real_doc.exists():
    doc_path = real_doc
    print(f"Using real document: {doc_path}")
else:
    doc_path = make_synthetic_document()
    print(f"Using synthetic document: {doc_path}")

result = detect_signatures(doc_path, conf_threshold=0.25)
print(f"\nDetected {result['count']} signature(s)")
show_detections(result)

## Demo 2 — Crop Detected Signatures for Further Analysis

The cropped signatures can be fed directly into Lab 03 (Signature Verification):


In [ ]:
crops_dir = Path("../data/samples/detected_crops")

if result["count"] > 0:
    saved_paths = crop_signatures(result, crops_dir)

    # Show crops
    fig, axes = plt.subplots(1, len(saved_paths), figsize=(4 * len(saved_paths), 3))
    if len(saved_paths) == 1:
        axes = [axes]
    for ax, path in zip(axes, saved_paths):
        ax.imshow(Image.open(path))
        ax.set_title(path.name, fontsize=9)
        ax.axis('off')
    plt.suptitle("Extracted Signature Crops", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"\n{len(saved_paths)} signature(s) saved to {crops_dir}")
    print("These can be used as input for Lab 03 (Signature Verification).")
else:
    print("No signatures detected — nothing to crop.")
    print("Try lowering conf_threshold or using a document with visible signatures.")

## Demo 3 — Load Your Own Document


In [ ]:
# ─── Change this path to your own scanned document ────────────────────────────
USER_DOC_PATH = "../data/samples/document_with_signature_01.png"
CONF_THRESHOLD = 0.30
# ──────────────────────────────────────────────────────────────────────────────

path = Path(USER_DOC_PATH)
if path.exists():
    res = detect_signatures(path, conf_threshold=CONF_THRESHOLD)
    print(f"Detected {res['count']} signature(s)")
    show_detections(res, title=path.name)
else:
    print(f"File not found: {path}")
    print("Place a scanned document image at the path above and re-run.")

## Forensic Notes

- Adjust **`conf_threshold`** based on your document quality. Low-resolution scans may need a lower threshold.
- For multi-page PDFs, convert each page to an image first (e.g., with `pdf2image` / `poppler`).
- Detected regions should be **reviewed by a human** before proceeding to automated verification — the detector may occasionally flag stamps, logos, or printed signatures.
- Detection output (coordinates, confidence scores) should be logged and preserved as part of the forensic record.
- Conditional DETR does **not** require Non-Maximum Suppression post-processing — each object query produces at most one prediction, eliminating duplicate detections by design.

---

**Next lab →** [05 — Writer Identification](05_writer_identification.ipynb)
